In [1]:
import rasterio as rio
import numpy as np
import os
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler 

# Explanatary Analysis

In [2]:
def load_data():
    print('Preprocessing  LISFLOOD data')
    #Preprocess water depth data
    lisflood_simulation_dir = "/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data/simulation_output"

    # Need to remove first 8 files from each run as they are not useful. A run is a simulation of a flood event. 
    # File start with Run2-0000.wd, Run2-0001.wd, Run2-0002.wd, Run2-0003.wd, Run2-0004.wd, Run2-0005.wd, Run2-0006.wd, Run2-0007.wd. There is Run2 to Run9 in the simulation output.
    inundation_files = {}
    for i in range(1, 10):
        inun_files = [file for file in os.listdir(lisflood_simulation_dir) if file.endswith('.wd') and file.startswith(f"Run{i}")]
        inundation_files[f"Run{i}"] = inun_files if len(inun_files) > 0 else []
    
    target = []
    test_target = []
    # Iterate the inundation_files dictionary
    for key, inun_files in inundation_files.items():
        inun_files.sort()
        # inun_files = inun_files[8:] # remove first 8 files
        for i in range(len(inun_files)):
            data = rio.open(os.path.join(lisflood_simulation_dir, inun_files[i]))
            band = data.read(1)
            # value = band.flatten()
            if key == 'Run1':
                test_target.append(band)
                continue
            target.append(band)

    Y_train = np.array(target)
    Y_test = np.array(test_target)
    Y_train[Y_train<0.3] = 0
    Y_test[Y_test<0.3] = 0

    #preprocess input features
    #Import Precipitation/Discharge Data
    file_dir = "/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data/"
    csv_files = [file for file in os.listdir(file_dir) if file.endswith('.csv')]

    #iterate the csv_files
    csv_files.sort()
    df_train = None   
    for i in range(len(csv_files)):
        if i == 0:
            continue
        df = pd.read_csv(os.path.join(file_dir, csv_files[i]))
        #remove the first 8 rows as it corressponds to frist two hours of simulation
        df = df.iloc[8:]
        if df_train is not None:
            df_train = pd.concat([df_train, df], ignore_index=True)
        else:
            df_train = df

    #Add test data at the end
    df_test = pd.read_csv(os.path.join(file_dir, csv_files[0]))
    df_test = df_test.iloc[8:]
    df_train = pd.concat([df_train, df_test], ignore_index=True)
    return df_train, Y_train, Y_test


def preprocess_data(df_train):
    print('Preprocessing data')
    scaler = MinMaxScaler()
    # Scale the data
    scaler_columns = ["Upstream1", "Upstream2", "Upstream3"]
    df_scaled = df_train.copy()
    df_scaled[scaler_columns] = scaler.fit_transform(df_train[scaler_columns])

    # Add lag features for 8 days
    for i in range(1, 10):
        for col in [c for c in df_train.columns if c != 'Time']:
            df_scaled[f'lag_{i}_{col}'] = df_scaled[col].shift(i)
            df_scaled.dropna(inplace=True)
    return df_scaled

def split_train_test_data(df_train):
    # Split the data into training and testing
    # Upstream_Flows_Run1 is used as the test data which has 274 time steps. So need to substract 274 from the length of training data
    X_train = df_train.iloc[:-266]
    X_test = df_train.iloc[-266:]

    X_train= X_train.drop(columns=['Time'])
    X_test = X_test.drop(columns=['Time'])
    #rehsape the data
    X_train = X_train.values.reshape(X_train.shape[0], 1, X_train.shape[1])
    X_test = X_test.values.reshape(X_test.shape[0], 1, X_test.shape[1])
    return X_train, X_test

In [1]:
def visualize_dem(file_path):
    with rio.open(file_path) as src:
        # Read the data
        data = src.read(1)  # Read the first band (elevation values)
        transform = src.transform  # Affine transform

    # Get the row, col indices of the array
    rows, cols = np.indices(data.shape)

    # Convert row, col to geographic coordinates
    xs, ys = rio.transform.xy(transform, rows, cols, offset='center')

    # Flatten the arrays for DataFrame creation
    xs_flat = np.array(xs).flatten()
    ys_flat = np.array(ys).flatten()
    data_flat = data.flatten()

    # Create a DataFrame
    df = pd.DataFrame({
        'x': xs_flat,
        'y': ys_flat,
        'elevation': data_flat
    })

    # Remove NoData values if needed (usually represented as a specific value, e.g., -9999)
    df = df[df['elevation'] != src.nodata]  # Adjust src.nodata if your NoData value is different

    # Save to CSV if needed
    df.to_csv('dem_dataframe.csv', index=False)

    # PLot the data on a map
    # Create a GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['x'], df['y']))

    # Plot the data
    fig, ax = plt.subplots(figsize=(10, 10))
    gdf.plot(column='elevation', ax=ax, legend=True, cmap='terrain')
    plt.show()

file_path = '/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data/Carlisle_5m.asc'
visualize_dem(file_path)

NameError: name 'rio' is not defined

In [5]:
def preprocess_data(df):
    print('Preprocessing data')
    scaler = MinMaxScaler()
    # Scale the data
    scaler_columns = ["Upstream1", "Upstream2", "Upstream3"]
    df_scaled = df.copy()
    df_scaled[scaler_columns] = scaler.fit_transform(df[scaler_columns])

    # Add lag features for 8 days
    # for i in range(1, 10):
    #     for col in [c for c in df.columns if c != 'Time']:
    #         df_scaled[f'lag_{i}_{col}'] = df_scaled[col].shift(i)
    #         df_scaled.dropna(inplace=True)
    return df_scaled

def load_elevation_data():
    file_path = '/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data/Carlisle_5m.asc'
    with rasterio.open(file_path) as src:
        # Read the data
        data = src.read(1)
        transform = src.transform
        rows, cols = np.indices(data.shape)
        rows = rows.flatten()
        cols = cols.flatten()
        id = [f"{r}_{c}" for r, c in zip(rows, cols)]
        xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')
        xs_flat = np.array(xs).flatten()
        ys_flat = np.array(ys).flatten()
        data_flat = data.flatten()
        df = pd.DataFrame({
            'id': id,
            'x': xs_flat,
            'y': ys_flat,
            'elevation': data_flat
        })

    return df


def load_inundation_data():
    print('Loading LISFLOOD data')

    # Preprocess water depth data
    lisflood_simulation_dir = "/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data/simulation_output"

    # Need to remove first 8 files from each run as they are not useful. A run is a simulation of a flood event. 
    # File start with Run2-0000.wd, Run2-0001.wd, Run2-0002.wd, Run2-0003.wd, Run2-0004.wd, Run2-0005.wd, Run2-0006.wd, Run2-0007.wd.
    #  There is Run2 to Run9 in the simulation output.
    inundation_files = {}
    for i in range(1, 10):
        print(f'Processing Run{i}')
        inun_files = [file for file in os.listdir(lisflood_simulation_dir) if file.endswith('.wd') and file.startswith(f"Run{i}")]
        inundation_files[f"Run{i}"] = inun_files if len(inun_files) > 0 else []
    
    target_data = []
    test_target = []
    # Iterate the inundation_files dictionary
    for key, inun_files in inundation_files.items():
        inun_files.sort()
        print(f'Processing {key} with {len(inun_files)} files')
        # inun_files = inun_files[8:]
        for i in range(len(inun_files)):
            timestep = key + "_" + str(i)
            data = rio.open(os.path.join(lisflood_simulation_dir, inun_files[i]))
            values = data.read(1)
            rows, cols = np.indices(values.shape)
            rows = rows.flatten()
            cols = cols.flatten()
            depth = values.flatten()
            cell_ids = [f"{r}_{c}" for r, c in zip(rows, cols)]
            df = pd.DataFrame({ 'id': cell_ids, 'depth': depth})
            df['timestep'] = timestep
            # If depth of water is less than 0.3m, set it to 0
            df.loc[df['depth'] < 0.3, 'depth'] = 0
            # if key == 'Run1':
            #     test_target = df
            #     continue
            target_data.append(df)

    # Concatenate the train_target list
    target_data = pd.concat(target_data, axis=0, ignore_index=True)
    return target_data

def load_boundry_condition_data():
    # Preprocess input features
    # Import Precipitation/Discharge Data
    file_dir = "/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data"
    csv_files = [file for file in os.listdir(file_dir) if file.endswith('.csv')]

    # Iterate the csv_files
    csv_files.sort()
    bc_data = None   
    for i in range(len(csv_files)):
        df = pd.read_csv(os.path.join(file_dir, csv_files[i]))
        # df = preprocess_data(df)
        # drop first 8 rows as it corressponds to frist two hours of simulation in which the lisflood model is initialising
        # df = df.iloc[8:]

        #temporarliy get the first five rows for testing
        df = df.iloc[:1]
        df['timestep'] = ["Run" + str(i+1) + "_" + str(idx) for idx in df.index]
        if bc_data is not None:
            bc_data = pd.concat([bc_data, df], axis=0, ignore_index=True)
        else:
            bc_data = df
    return bc_data

def merge_data(bc_data, elevation):
    bc_data['key'] = 1
    elevation['key'] = 1
    train_df = pd.merge(bc_data, elevation, on='key')
    train_df = train_df.drop(columns=['key'])
    # train_df = pd.merge(target_data, train_df, on='timestep')
    return train_df

def create_seqences(merged_data, time_steps, forecast_horizon, excluded_columns):
    #create sequences of  (timesteps, features) for each timestep
    X, y = [], []
    for cell_id in merged_data['cell_id'].unique():
        cell_data = merged_data[merged_data['cell_id'] == cell_id].sort_values('timestep')
        target_values = cell_data['depth'].values
        
        #Remove excluded columns
        cell_data = cell_data.drop(excluded_columns, axis=1).astype(float)
        cell_data_values = cell_data.values

        for i in range(len(cell_data_values) - time_steps - forecast_horizon):
            X.append(cell_data_values[i:i+time_steps])
            y.append(target_values[i+time_steps:i+time_steps+forecast_horizon])
    return np.array(X), np.array(y)

# Batch Processing

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, LSTM
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model
import timeit
import matplotlib.pyplot as plt

def LSTM_Model(timesteps, features, forecast_horizon):
    '''
    LSTM model
    '''
    model = Sequential()
    model.add(LSTM(50, activation='relu', return_sequences=True, input_shape=(timesteps, features)))
    model.add(LSTM(50, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(256, activation='relu'))
    model.add(Dense(512, activation='relu'))
    model.add(Dense(forecast_horizon))
    optimizer = Adam(learning_rate=0.01)
    model.compile(loss='mse', metrics=['mse'], optimizer=optimizer)
    print(model.summary())
    return model

def train_model(generator, model):
    print("Training the model for batch number: ")
    monitor = EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=5, verbose=1, mode='auto')
    history = model.fit(generator,batch_size=10, callbacks=[monitor], verbose=0, epochs=100)
    return model, history

def plot_loss(history):
    # plot_model(model, to_file='/home/91/23016891/projects/Rapid_FloodModelling_CNN/LSTM_Graph.png', dpi=1200)
    # Plot history
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='test')
    plt.xlabel('epochs')
    plt.ylabel('loss')
    plt.legend()
    plt.show()

In [ ]:
import rasterio as rio

def load_data_batch(batch_size=100):
    print('Loading LISFLOOD data in batches')

    ## Boundry Condition Data
    bc_data_dir = "/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data"
    csv_files = [file for file in os.listdir(bc_data_dir) if file.endswith('.csv')]
    csv_files.sort()

    ## LISFLOOD Simulation Data
    lisflood_simulation_dir = "/Users/malintha/Documents/PHD/ARTEFACT1/TIME_SERIES_ML/Rapid_FloodModelling_CNN/Data/simulation_output"
    inundation_files = {}
    for i in range(1, 3):
        inun_files = [file for file in os.listdir(lisflood_simulation_dir) if file.endswith('.wd') and file.startswith(f"Run{i}")]
        inundation_files[f"Run{i}"] = inun_files if len(inun_files) > 0 else []
    inun_files.sort()

    # Elevation Data
    for key, inun_files in inundation_files.items():

        bc_file_name = [filename for filename in csv_files if key in filename][0]
        bc_df = pd.read_csv(os.path.join(bc_data_dir, bc_file_name))
        # bc_df = bc_df.iloc[8:]
        bc_df['timestep'] = ["Run" + str(csv_files.index(bc_file_name) + 1) + "_" + str(idx) for idx in bc_df.index]

        # inun_files = inun_files[8:]
        for i in range(0, len(inun_files), batch_size):
            batch_files = inun_files[i:i + batch_size]
            bc_df = bc_df.iloc[i:i + batch_size]
            batch_data = []
            for file in batch_files:
                timestep = key + "_" + str(i)
                data = rio.open(os.path.join(lisflood_simulation_dir, file))
                values = data.read(1)
                rows, cols = np.indices(values.shape)
                rows = rows.flatten()
                cols = cols.flatten()
                depth = values.flatten()
                cell_ids = [f"{r}_{c}" for r, c in zip(rows, cols)]
                df = pd.DataFrame({'id': cell_ids, 'depth': depth})
                df['timestep'] = timestep
                df.loc[df['depth'] < 0.3, 'depth'] = 0
                batch_data.append(df)
            target_df = pd.concat(batch_data, axis=0, ignore_index=True)
            print(target_df.shape)
            target_df['Upstream1'] = target_df['timestep'].map(bc_df.set_index('timestep')['Upstream1'])
            target_df['Upstream2'] = target_df['timestep'].map(bc_df.set_index('timestep')['Upstream2'])
            target_df['Upstream3'] = target_df['timestep'].map(bc_df.set_index('timestep')['Upstream3'])
            yield target_df


# def train_model_in_batches(batch_size=100):
#     generator = load_data_batch(batch_size)
#     batch_data = next(generator)
    
#     model = None
#     timesteps = 8
#     forecast_horizon = 4

#     start = timeit.default_timer()
#     model = LSTM_Model(timesteps, features, forecast_horizon)
#     for idx, batch_data in enumerate(load_data_batch(batch_size)):
#         # feature_and_traget_data = merge_data(bc_data, elevation, batch_data)
#         print(f'Processing batch number {idx}')
#         print(f'Feature and target data shape: {batch_data.shape}')
#         X, Y = create_seqences(batch_data, 7, 1, ['timestep', 'id', 'cell_id', 'depth'])
#         if model is None:
#                 timesteps = 8
#                 features = X.shape[2]
                
#         model, history = train_model(model, X_train, Y_train, X_test, Y_test)
#         plot_loss(history)
#     stop = timeit.default_timer()
#     print('Time: ', stop - start)


In [12]:
# Get the next batch from the generator
generator = load_data_batch(batch_size=1)
batch_data = next(generator)
batch_data.head()
# batch_data["key"] = 1
# bc_df["key"] = 1
# merged = batch_data.merge(bc_df, on='key')


# Now merge the batch data and bc_df


# Print the shapes of the returned data
# print("Batch data shape:", batch_data.shape)
# print("Boundary condition data shape:", bc_df.shape)
# print( "Merged", merged.shape)

Loading LISFLOOD data in batches
round
(581061, 3)


,id,depth,timestep,Upstream1,Upstream2,Upstream3
0,0_0,0.0,Run1_0,1.0,0.12,1.2
1,0_1,0.0,Run1_0,1.0,0.12,1.2
2,0_2,0.0,Run1_0,1.0,0.12,1.2
3,0_3,0.0,Run1_0,1.0,0.12,1.2
4,0_4,0.0,Run1_0,1.0,0.12,1.2
